# 2 - QLoRA fine-tune (RAFT) on a T4, then merge the adapter

Kaggle: GPU T4 x2 (only GPU 0 is used), Internet on, notebook 1's output added as an input.

Setup: Qwen2.5-1.5B-Instruct in 4-bit NF4, LoRA r=32 on all linear layers, loss on the reply only. The T4 has no bf16, so everything runs in fp16.

Time: ~8k examples x 2 epochs is a few hours on one T4 (the logs show the real tokens/s). Kaggle caps a session at 12 h and ~30 GPU-h a week, so checkpoints are saved every 200 steps and training runs with `--resume`. If the session dies, save a version with the partial output and re-run, or just re-run in the same session.

Run the 10-step smoke test first - it catches an OOM in a couple of minutes instead of a couple of hours.

Where this fits: notebook 1 built the RAFT training set, this one turns it into a model, notebook 3 compares it against plain RAG.

In [ ]:
!pip -q install "trl==0.24.0" "transformers>=4.56.1" peft bitsandbytes accelerate datasets pyyaml
import os; os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import os
os.makedirs('/kaggle/working/proj/configs', exist_ok=True)
os.makedirs('/kaggle/working/proj/src', exist_ok=True)
%cd /kaggle/working/proj

In [ ]:
%%writefile /kaggle/working/proj/configs/config.yaml
# RAFT vs RAG on a single T4. The same config is written in all three notebooks so every stage reads the same numbers.
seed: 42

data:
  doc2dial_url: https://doc2dial.github.io/file/doc2dial_v1.0.1.zip
  holdout_doc_frac: 0.10     # docs never seen in training -> "unseen-doc" test set
  passage_words: 140         # target passage size when chunking sections
  history_turns: 4           # previous turns shown in the prompt
  max_train: 8000
  max_valid: 200
  max_test_seen: 500
  max_test_unseen: 300
  valid_dialogue_frac: 0.04

retrieval:
  embed_model: BAAI/bge-small-en-v1.5
  query_prefix: "Represent this sentence for searching relevant passages: "
  save_top: 20               # stored per query (hard negatives + recall@k)
  k: 4                       # docs given to the model at test time

raft:
  n_docs: 4                  # docs in each training prompt
  p_oracle: 0.8              # gold passage is in the context 80% of the time; the RAFT paper found 60-100% all work
  hard_neg_frac: 0.7         # share of distractors taken from retriever top-10
  style: cot                 # cot = quote then answer | direct = answer only
  max_tokens: 1900           # drop longer examples so nothing gets truncated mid-answer at max_length 2048

train:
  base_model: Qwen/Qwen2.5-1.5B-Instruct     # stretch: Qwen/Qwen2.5-3B-Instruct
  max_length: 2048
  epochs: 2
  per_device_batch: 2        # x grad_accum 8 = effective batch 16. use 1 for the 3B model
  grad_accum: 8
  lr: 2.0e-4
  warmup_ratio: 0.03
  lora_r: 32
  lora_alpha: 64             # alpha = 2r, the usual starting point
  lora_dropout: 0.05
  save_steps: 200
  eval_steps: 200

eval:
  max_model_len: 4096
  gpu_util: 0.88             # share of VRAM vLLM grabs up front (weights + KV cache)
  max_new_tokens_direct: 256
  max_new_tokens_cot: 384

bench:
  n_queries: 200
  warmup: 10
  # rough on-demand T4 price, not a quote. cost per 1k queries is linear in it, so it's easy to redo
  gpu_usd_per_hour: 0.35

In [ ]:
%%writefile /kaggle/working/proj/src/common.py
"""Shared helpers: config, IO, prompts, answer parsing, metrics, retriever.

Everything that must be identical between the RAG baseline and the RAFT model
(prompt template, retrieval query, metrics) lives here so the comparison is fair.
"""
from __future__ import annotations

import json
import random
import re
import string
from collections import Counter
from pathlib import Path


# ----------------------------------------------------------------------------- IO
# read the yaml config into a plain dict - every script starts with this
def load_cfg(path: str = "configs/config.yaml") -> dict:
    import yaml

    with open(path) as f:
        return yaml.safe_load(f)


# fix every random source we use (python, numpy, torch) so reruns give the same splits and samples
def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch

        torch.manual_seed(seed)
    except ImportError:
        pass


# one JSON object per line -> list of dicts (blank lines skipped)
def read_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


# opposite of read_jsonl; creates the folder if it doesn't exist yet
def write_jsonl(path, rows) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


# save a single object (stats, reports) as pretty-printed JSON
def write_json(path, obj) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------------------ prompts
# system prompt when documents are given. the 'say so if the docs don't have it' line matters:
# RAFT trains with missing gold passages, so the model should learn to admit it
SYSTEM = (
    "You are a customer-support agent. Use the reference documents to answer the "
    "customer's last message. Be accurate and concise, and do not invent policies "
    "or numbers. If the documents do not contain the answer, say so."
)
# closed-book version, used when there are no documents in the prompt
SYSTEM_CLOSED = "You are a customer-support agent. Answer the customer's last message accurately and concisely."
# extra instruction for the cot style: quote the evidence first, then answer
COT_INSTR = (
    "First copy the sentences you rely on between ##begin_quote## and ##end_quote##, "
    "then write the reply after ##Answer:"
)


# turn the conversation into the search query used by the retriever
def retrieval_query(history: list[dict]) -> str:
    """Dialogue-aware query: the last two customer turns."""
    # a follow-up like "how do I do that?" means nothing on its own, so the previous customer turn comes along
    users = [t["text"] for t in history if t["role"] == "user"]
    return " ".join(users[-2:])


# one passage as the model sees it: [n] title - section, then the text
def fmt_passage(i: int, p: dict) -> str:
    head = p["title"] + (f" - {p['section']}" if p.get("section") else "")
    return f"[{i}] {head}\n{p['text']}"


# build the chat prompt: documents (if any) + conversation so far + what to write.
# used for training AND evaluation, so RAFT is tested on exactly the format it learned
def build_messages(history: list[dict], passages: list[dict], style: str = "direct") -> list[dict]:
    parts = []
    if passages:
        parts.append("Reference documents:\n" + "\n\n".join(fmt_passage(i + 1, p) for i, p in enumerate(passages)))
    conv = "\n".join(f"{'Customer' if t['role'] == 'user' else 'Agent'}: {t['text']}" for t in history)
    parts.append("Conversation so far:\n" + conv)
    parts.append("Write the agent's next reply." + (" " + COT_INSTR if style == "cot" else ""))
    return [
        {"role": "system", "content": SYSTEM if passages else SYSTEM_CLOSED},
        {"role": "user", "content": "\n\n".join(parts)},
    ]


# the text the model is trained to produce: quote + answer (cot) or just the answer (direct)
def build_target(style: str, quote: str, response: str) -> str:
    # doc2dial already gives the grounding span, so the quote half of the target is free (no teacher model)
    if style == "cot":
        return f"##Reason: The relevant text says ##begin_quote## {quote} ##end_quote##.\n##Answer: {response}"
    return response


# grabs whatever sits between the quote markers
_QUOTE_RE = re.compile(r"##begin_quote##(.*?)##end_quote##", re.S)


# split a model output into the final answer and its quotes - only the answer gets scored
def parse_output(text: str) -> tuple[str, list[str]]:
    """Return (final answer, quoted spans). Works for direct and cot outputs."""
    quotes = [q.strip() for q in _QUOTE_RE.findall(text)]
    # base models never write ##Answer:, so fall back to the whole text minus the markers
    if "##Answer:" in text:
        ans = text.rsplit("##Answer:", 1)[1]
    else:
        ans = _QUOTE_RE.sub("", text).replace("##Reason:", "")
    return ans.strip(), quotes


# ------------------------------------------------------------------------ metrics
# articles and stop words are ignored when comparing texts
_ARTICLES = re.compile(r"\b(a|an|the)\b")
_STOP = set(
    "a an the and or but if of to in on for with at by from as is are was were be been it this that these those "
    "you your i we our they their he she his her not no do does did can will would should could may might have has had "
    "so than then there here what which who whom when where how about into over under up out".split()
)


# clean a string before comparing, so 'The SSA.' and 'ssa' count as the same
def normalize(s: str) -> str:
    # SQuAD-style: lowercase, drop punctuation and articles
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


# word-overlap F1 between our reply and the human agent's reply (0 = nothing shared, 1 = same words)
def token_f1(pred: str, gold: str) -> float:
    p, g = normalize(pred).split(), normalize(gold).split()
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    n = sum(common.values())
    if n == 0:
        return 0.0
    prec, rec = n / len(p), n / len(g)
    return 2 * prec * rec / (prec + rec)


# like F1 but based on the longest common subsequence, so word order counts a bit
def rouge_l(pred: str, gold: str) -> float:
    # plain LCS DP; the 200-token cap keeps it cheap when a model rambles
    p, g = normalize(pred).split()[:200], normalize(gold).split()[:200]
    if not p or not g:
        return float(p == g)
    prev = [0] * (len(g) + 1)
    for a in p:
        cur = [0]
        for j, b in enumerate(g):
            cur.append(prev[j] + 1 if a == b else max(prev[j + 1], cur[j]))
        prev = cur
    lcs = prev[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(g)
    return 2 * prec * rec / (prec + rec)


# how much of the answer is backed by the documents we passed in - low value = likely made up
def groundedness(answer: str, context: str) -> float:
    """Share of content words in the answer that appear in the supplied context (hallucination proxy)."""
    # crude on purpose: word overlap, not entailment. good for comparing models side by side,
    # not as an absolute hallucination rate
    a = [w for w in normalize(answer).split() if w not in _STOP]
    if not a:
        return 1.0
    c = set(normalize(context).split())
    return sum(w in c for w in a) / len(a)


# did the model quote text that really exists in the context, or invent it?
def quote_grounded(quotes: list[str], context: str) -> float | None:
    """Share of quoted spans that really occur in the context (None if no quotes)."""
    # only the cot model quotes - this checks it isn't inventing them
    if not quotes:
        return None
    c = normalize(context)
    return sum(normalize(q) in c for q in quotes if normalize(q)) / max(1, sum(1 for q in quotes if normalize(q)))


# p50 / p95 for latency numbers (linear interpolation, same idea as numpy.percentile)
def percentile(xs: list[float], q: float) -> float:
    xs = sorted(xs)
    if not xs:
        return float("nan")
    k = (len(xs) - 1) * q / 100
    lo, hi = int(k), min(int(k) + 1, len(xs) - 1)
    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)


# ---------------------------------------------------------------------- retriever
# dense retriever: embed all passages once, then a query is one matrix multiply + top-k.
# the exact same object is used for building RAFT data, RAG eval and the latency benchmark
class Retriever:
    """Dense retriever (bge-small by default). Whole corpus is a few thousand passages -> brute force matmul."""

    # load the embedding model and embed the corpus (or load the cached embeddings)
    def __init__(self, passages: list[dict], embed_model: str, prefix: str, device: str | None = None, emb_path=None):
        import numpy as np
        import torch
        from sentence_transformers import SentenceTransformer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.prefix = prefix
        self.passages = passages
        self.model = SentenceTransformer(embed_model, device=self.device)
        if self.device == "cuda":
            self.model.half()
        # embeddings are cached to disk so eval / bench don't re-embed the corpus every time
        if emb_path and Path(emb_path).exists():
            emb = torch.from_numpy(np.load(emb_path))
        else:
            texts = [f"{p['title']} - {p['section']}\n{p['text']}" for p in passages]
            emb = self.model.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=True).float().cpu()
            if emb_path:
                Path(emb_path).parent.mkdir(parents=True, exist_ok=True)
                np.save(emb_path, emb.numpy())
        self.emb = emb.to(self.device).half() if self.device == "cuda" else emb

    # top-k passage indices + similarity scores for many queries, in batches of `bs`
    def search_batch(self, queries: list[str], k: int, bs: int = 128) -> tuple[list[list[int]], list[list[float]]]:
        idxs, scores = [], []
        for i in range(0, len(queries), bs):
            q = self.model.encode([self.prefix + x for x in queries[i : i + bs]], normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=False)
            sims = q.to(self.emb.dtype) @ self.emb.T
            s, ix = sims.topk(min(k, sims.shape[1]), dim=1)
            idxs += ix.cpu().tolist()
            scores += s.float().cpu().tolist()
        return idxs, scores

    # single-query version, used in the benchmark where queries arrive one by one
    def search(self, query: str, k: int) -> list[int]:
        return self.search_batch([query], k)[0][0]

In [ ]:
%%writefile /kaggle/working/proj/src/train_qlora.py
"""Stage 4 - QLoRA fine-tuning on one Kaggle T4 (16 GB, fp16 only: no bf16, no FlashAttention-2).

Recipe: 4-bit NF4 base + LoRA(r=32, all linear layers) + fp16 autocast + gradient checkpointing +
paged 8-bit AdamW, loss on the completion only (TRL prompt-completion format).

    python src/train_qlora.py --data_dir work/data/raft_cot --out_dir work/ckpt/raft_cot
    python src/train_qlora.py ... --resume          # Kaggle sessions die: re-run to continue from last checkpoint
"""
from __future__ import annotations

import argparse
import time
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer

from common import load_cfg, set_seed, write_json


# small trainer hook: notes the start time and resets the peak-VRAM counter right before training,
# so the VRAM number in the report only covers training itself
class Speed(TrainerCallback):
    def __init__(self):
        self.t0 = None

    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
        torch.cuda.reset_peak_memory_stats()


# load base in 4-bit -> attach LoRA -> train on the prompt/completion data -> save adapter + report
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--data_dir", required=True, help="folder with train.jsonl / valid.jsonl in prompt-completion format")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--model", default=None)
    ap.add_argument("--epochs", type=float, default=None)
    ap.add_argument("--bs", type=int, default=None)
    ap.add_argument("--accum", type=int, default=None)
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--max_steps", type=int, default=-1, help="smoke test: e.g. 20")
    ap.add_argument("--resume", action="store_true")
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    t = cfg["train"]
    model_id = a.model or t["base_model"]
    set_seed(cfg["seed"])
    assert torch.cuda.is_available(), "needs a GPU (Kaggle: Settings -> Accelerator -> GPU T4)"

    # tokenizer; if it has no pad token, fall back to EOS so batching works
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # 4-bit base model: 1.5B weights shrink to ~1 GB, leaving room for activations on 16 GB
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)  # T4: fp16 compute, never bf16
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map={"": 0},
                                                 torch_dtype=torch.float16, attn_implementation="sdpa")
    model.config.use_cache = False  # KV cache is useless while training and clashes with gradient checkpointing

    # all-linear = every attention and MLP projection, same as the QLoRA paper
    lora = LoraConfig(r=t["lora_r"], lora_alpha=t["lora_alpha"], lora_dropout=t["lora_dropout"],
                      target_modules="all-linear", bias="none", task_type="CAUSAL_LM")

    dd = Path(a.data_dir)
    ds = load_dataset("json", data_files={"train": str(dd / "train.jsonl"), "valid": str(dd / "valid.jsonl")})

    args = SFTConfig(
        output_dir=a.out_dir,
        num_train_epochs=a.epochs or t["epochs"],
        max_steps=a.max_steps,
        per_device_train_batch_size=a.bs or t["per_device_batch"],
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=a.accum or t["grad_accum"],
        learning_rate=a.lr or t["lr"],
        lr_scheduler_type="cosine",
        warmup_ratio=t["warmup_ratio"],
        weight_decay=0.0,
        max_grad_norm=1.0,
        fp16=True, bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",          # paged 8-bit states ride out memory spikes on 16 GB
        max_length=t["max_length"],
        completion_only_loss=True,          # loss only on the assistant reply
        packing=False,
        logging_steps=10,
        eval_strategy="steps", eval_steps=t["eval_steps"],
        save_strategy="steps", save_steps=t["save_steps"], save_total_limit=2,
        report_to="none",
        seed=cfg["seed"],
        dataloader_num_workers=2,
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["valid"],
                         processing_class=tok, peft_config=lora, callbacks=[Speed()])
    trainer.model.print_trainable_parameters()

    # --resume only if a checkpoint actually exists, otherwise start fresh
    ckpts = list(Path(a.out_dir).glob("checkpoint-*"))
    t0 = time.time()
    trainer.train(resume_from_checkpoint=True if (a.resume and ckpts) else None)
    wall = time.time() - t0

    # save the LoRA adapter (tiny next to the full model) and a report with losses, time and VRAM
    final = Path(a.out_dir) / "adapter"
    trainer.model.save_pretrained(final)
    tok.save_pretrained(final)
    logs = [l for l in trainer.state.log_history if "loss" in l]
    evals = [l for l in trainer.state.log_history if "eval_loss" in l]
    write_json(Path(a.out_dir) / "train_report.json", {
        "model": model_id, "train_examples": len(ds["train"]), "wall_minutes": round(wall / 60, 1),
        "steps": trainer.state.global_step, "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
        "final_train_loss": logs[-1]["loss"] if logs else None, "final_eval_loss": evals[-1]["eval_loss"] if evals else None,
        "effective_batch": args.per_device_train_batch_size * args.gradient_accumulation_steps,
        "log_history": trainer.state.log_history})
    print(f"adapter saved to {final}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/merge_adapter.py
"""Stage 5 - merge the LoRA adapter into an fp16 copy of the base model so vLLM can serve it as a plain model.

    python src/merge_adapter.py --base Qwen/Qwen2.5-1.5B-Instruct --adapter work/ckpt/raft_cot/adapter --out work/merged/raft_cot

Note: QLoRA trains against a 4-bit base but we merge into the fp16 base (standard practice). The tiny
quantisation mismatch is real; we evaluate the merged fp16 model, which is also what gets served.
"""
import argparse

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# plain script, no functions: load base -> apply adapter -> fold it in -> save as a normal model
ap = argparse.ArgumentParser()
ap.add_argument("--base", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
a = ap.parse_args()

# merged on CPU: slower, but it can't OOM on the T4
model = AutoModelForCausalLM.from_pretrained(a.base, torch_dtype=torch.float16, low_cpu_mem_usage=True, device_map={"": "cpu"})
model = PeftModel.from_pretrained(model, a.adapter).merge_and_unload()
model.save_pretrained(a.out, safe_serialization=True)
AutoTokenizer.from_pretrained(a.base).save_pretrained(a.out)
print("merged model saved to", a.out)

In [ ]:
import glob
# first file under /kaggle/input that matches the pattern; fails loudly if the input isn't attached
def find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"nothing matches {pattern} - did you add the previous notebook as an Input?"
    return hits[0]
DATA = os.path.dirname(find('/kaggle/input/**/raft_cot/train.jsonl'))
print('training data:', DATA)
import torch; print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info()[1] / 2**30, 'GB')

### Smoke test (10 steps)
Just confirms the model, data and batch size fit before the real run.

In [ ]:
!python src/train_qlora.py --data_dir {DATA} --out_dir work/ckpt/smoke --max_steps 10

### Full training - RAFT (quote-then-answer)
If it OOMs: `--bs 1 --accum 16` (same effective batch). If the loss goes `nan`: `--lr 1e-4`. For the 3B model: `--model Qwen/Qwen2.5-3B-Instruct --bs 1 --accum 16`.

In [ ]:
!python src/train_qlora.py --data_dir {DATA} --out_dir work/ckpt/raft_cot --resume

In [ ]:
import json; r = json.load(open('work/ckpt/raft_cot/train_report.json')); r.pop('log_history'); print(json.dumps(r, indent=1))

### Merge the adapter into the fp16 base
vLLM serves a plain model in notebook 3, so the LoRA weights get folded in here.

In [ ]:
!python src/merge_adapter.py --base Qwen/Qwen2.5-1.5B-Instruct --adapter work/ckpt/raft_cot/adapter --out work/merged/raft_cot

### Optional ablation - `direct` (answer-only) variant
Same training, target without the quote, merged as `raft_direct`. Not needed for the main comparison; skip it if GPU quota is tight.

In [ ]:
# optional ablation, commented out to save GPU quota
# DATA_D = os.path.dirname(find('/kaggle/input/**/raft_direct/train.jsonl'))
# !python src/train_qlora.py --data_dir {DATA_D} --out_dir work/ckpt/raft_direct --resume
# !python src/merge_adapter.py --base Qwen/Qwen2.5-1.5B-Instruct --adapter work/ckpt/raft_direct/adapter --out work/merged/raft_direct

Clean-up: drop the intermediate checkpoints so the saved output is just the adapter, the merged model and the logs.

In [ ]:
!rm -rf work/ckpt/smoke work/ckpt/*/checkpoint-*
!du -sh work/*